# Évaluation des capacités des modèles de langage à identifier des schèmes imagés

**Mémoire — Sujet 9**
Auteurs du sujet : François Olivier et Sophie Robert-Hayek

Ce notebook met en œuvre le protocole expérimental du mémoire :

1. Évaluer si des LLMs récents (GPT, Claude, Mistral, LLaMA, …) identifient correctement le
   **schème imagé** (*image schema*, Lakoff & Johnson 1980) sous-tendant une expression linguistique.
2. Comparer trois **stratégies de prompting** : explication libre, choix guidé (liste fermée),
   choix guidé avec justification.
3. S'appuyer sur deux ressources existantes :
   - `data/Image_Schemas_English_and_German.csv` — corpus annoté EN/DE (repris de
     Wachowiak & Gromann 2022/2023 ; catégories inspirées d'ImageSchemaNet, De Giorgis et al. 2022/2024).
   - `data/100_for_eval_fnroles_out.csv` — 98 expressions annotées (6 schèmes), avec les prédictions
     d'un système explicable antérieur basé FrameNet/WordNet (colonne `pred`, servant de **baseline**
     de comparaison), utile pour l'analyse des cas ambigus.
4. Analyser linguistiquement les résultats : expressions bien traitées, erreurs récurrentes,
   tendances interprétatives, accord inter-modèles.
5. Étudier les préférences des modèles face à l'ambiguïté schématique, notamment la question de la
   **directionnalité de SOURCE-PATH-GOAL appliqué au temps** (métaphore *moving time* vs *moving ego*).

> **Mode démo.** Ce notebook n'a pas vos clés d'API : il est conçu pour tourner **tel quel**, en
> `DEMO_MODE = True`, avec un modèle factice qui simule des réponses (pour vérifier que tout le
> pipeline fonctionne sans rien payer). Passez `DEMO_MODE = False` et renseignez vos clés (section
> Configuration) pour lancer les vraies évaluations sur GPT / Claude / Mistral / LLaMA.


## 1. Installation et imports

In [ ]:
# À exécuter une fois (Colab ou environnement local neuf).
# Les SDKs des fournisseurs ne sont nécessaires qu'en mode réel (DEMO_MODE = False) ;
# ils sont importés paresseusement plus bas pour ne pas bloquer le mode démo.
!pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm
!pip install -q openai anthropic mistralai  # ignorés si déjà installés / si offline


In [ ]:
import json
import os
import re
import time
import random
import getpass
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 2. Configuration

- `DEMO_MODE = True` (par défaut) : aucune clé requise, les appels aux modèles sont simulés par
  `mock_llm_call` (section 5) afin que tout le notebook s'exécute de bout en bout.
- Pour lancer une vraie évaluation, passez `DEMO_MODE = False` et renseignez au moins une clé
  d'API ci-dessous (variables d'environnement, ou saisie interactive via `getpass`).
- `MODELS_TO_EVALUATE` liste les modèles réellement interrogés. Mettez à jour les identifiants de
  modèle (`model=`) en fonction de ce qui est disponible au moment où vous lancez le notebook —
  les noms ci-dessous sont ceux visés par le sujet (GPT-5, Claude, Mistral, LLaMA) mais les
  identifiants exacts d'API évoluent.


In [ ]:
DEMO_MODE = True  # <-- passez à False pour lancer de vraies requêtes API

# Clés d'API (laissées vides en mode démo). Ne jamais committer de vraies clés dans le notebook.
os.environ.setdefault("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))
os.environ.setdefault("ANTHROPIC_API_KEY", os.environ.get("ANTHROPIC_API_KEY", ""))
os.environ.setdefault("MISTRAL_API_KEY", os.environ.get("MISTRAL_API_KEY", ""))
# Pour LLaMA : un point de terminaison compatible OpenAI (Groq, Together, Ollama local, ...)
os.environ.setdefault("LLAMA_API_BASE", os.environ.get("LLAMA_API_BASE", "https://api.groq.com/openai/v1"))
os.environ.setdefault("LLAMA_API_KEY", os.environ.get("LLAMA_API_KEY", ""))

if not DEMO_MODE:
    for key_name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "MISTRAL_API_KEY", "LLAMA_API_KEY"]:
        if not os.environ.get(key_name):
            try:
                os.environ[key_name] = getpass.getpass(f"{key_name} (laisser vide pour ignorer ce fournisseur) : ")
            except Exception:
                pass

# Déclaration des modèles à évaluer : (nom_affiché, fournisseur, identifiant_modele)
# fournisseur ∈ {"openai", "anthropic", "mistral", "openai_compatible"}
MODELS_TO_EVALUATE = [
    {"name": "GPT-5",          "provider": "openai",            "model": "gpt-5"},
    {"name": "Claude",         "provider": "anthropic",         "model": "claude-sonnet-5"},
    {"name": "Mistral Large",  "provider": "mistral",           "model": "mistral-large-latest"},
    {"name": "LLaMA",          "provider": "openai_compatible",  "model": "llama-3.3-70b-versatile",
     "base_url": os.environ["LLAMA_API_BASE"], "api_key_env": "LLAMA_API_KEY"},
]

print("Mode démo :", DEMO_MODE)
print("Modèles configurés :", [m["name"] for m in MODELS_TO_EVALUATE])


## 3. Chargement des données

Les deux fichiers sont attendus dans `data/` (à la racine du dépôt). Sur Colab sans le dépôt
cloné, la cellule ci-dessous propose un import manuel de secours.


In [ ]:
DATA_DIR = Path("data")

def _load_csv_with_fallback(filename):
    path = DATA_DIR / filename
    if path.exists():
        return pd.read_csv(path)
    # Repli Colab : upload manuel si le fichier n'est pas présent localement
    try:
        from google.colab import files  # type: ignore
        print(f"'{filename}' introuvable dans data/. Merci de l'uploader :")
        uploaded = files.upload()
        return pd.read_csv(list(uploaded.keys())[0])
    except ImportError:
        raise FileNotFoundError(
            f"'{path}' introuvable. Placez le fichier dans le dossier data/ du dépôt."
        )

df_schemas = _load_csv_with_fallback("Image_Schemas_English_and_German.csv")
df_eval_fn = _load_csv_with_fallback("100_for_eval_fnroles_out.csv")

print("Image_Schemas_English_and_German :", df_schemas.shape)
print("100_for_eval_fnroles_out         :", df_eval_fn.shape)
df_schemas.head(3)


In [ ]:
df_eval_fn.head(3)

## 4. Exploration rapide des deux corpus

But : comprendre la distribution des schèmes avant de construire l'ensemble de test soumis aux LLMs.


In [ ]:
# Dédoublonnage (une même expression peut apparaître pour plusieurs schèmes/langues)
df_schemas = df_schemas.drop_duplicates(subset=["LinguisticExamples", "IMAGE_SCHEMA_ANNOTATION"]).reset_index(drop=True)

print("Répartition par langue :")
print(df_schemas["Language"].value_counts())
print()
print("Répartition par schème (IMAGE_SCHEMA_ANNOTATION) :")
print(df_schemas["IMAGE_SCHEMA_ANNOTATION"].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_schemas["IMAGE_SCHEMA_ANNOTATION"].value_counts().plot(kind="barh", ax=axes[0], color="#4C72B0")
axes[0].set_title("Corpus 1 — Image_Schemas_EN_DE : distribution des schèmes")
axes[0].invert_yaxis()

df_eval_fn["label"].value_counts().plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title("Corpus 2 — 100_for_eval_fnroles : distribution des schèmes (gold)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 5. Taxonomie commune des schèmes imagés

Les deux corpus n'utilisent pas exactement la même granularité (14 catégories dans le corpus 1,
6 dans le corpus 2). On construit une **liste fermée** unique, avec une définition courte pour
chaque schème, utilisée pour le prompting en *choix guidé* (stratégies B et C). Une option
`OTHER` est prévue pour ne pas forcer un choix inadapté — utile pour l'analyse des cas limites.


In [ ]:
SCHEMA_DEFINITIONS = {
    "CONTAINMENT":      "Being inside vs. outside a bounded region (in / out, container / contained).",
    "SOURCE_PATH_GOAL": "Motion or extension from a starting point (source), along a trajectory (path), to an endpoint (goal).",
    "PART_WHOLE":       "An entity composed of parts, or a part belonging to / detached from a whole.",
    "CENTER_PERIPHERY": "A central, core area contrasted with an outer, less prominent area.",
    "SUPPORT":          "One entity holding up, propping up, or bearing the weight of another.",
    "BLOCKAGE":         "An obstacle or barrier that prevents or impedes motion or progress.",
    "OBJECT":           "An abstract entity conceived of as a discrete, manipulable physical object.",
    "VERTICALITY":      "Orientation along a vertical axis (up/down), where up is often MORE and down is often LESS.",
    "FORCE":            "A physical or causal force exerted, resisted, or transmitted between entities.",
    "SUBSTANCE":        "An abstract entity conceived of as an unbounded mass or material.",
    "SCALE":            "A graded ordering of quantity or intensity along a single dimension.",
    "CONTACT":          "Direct touching or physical connection between two entities.",
    "SPLITTING":        "A single entity separating or dividing into distinct parts.",
    "COVERING":         "One entity spread over another, hiding or protecting it.",
    "LINK":             "A connection or bond joining two otherwise separate entities.",
    "NEAR_FAR":         "Relative distance / proximity between two entities.",
    "FRONT_BACK":       "Orientation along a front/back axis, often tied to the direction of motion.",
    "LOCOMOTION":       "Self-propelled movement of an entity through space.",
    "OTHER":            "None of the above fits well — specify what schema you would use instead.",
}

def normalize_schema_label(raw):
    """Normalise une étiquette de schème vers MAJUSCULES_SNAKE_CASE, avec quelques synonymes usuels."""
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    s = str(raw).strip().upper().replace("-", "_").replace(" ", "_")
    synonyms = {
        "CONTAINER": "CONTAINMENT",
        "UP_DOWN": "VERTICALITY",
        "UP-DOWN": "VERTICALITY",
    }
    s = synonyms.get(s, s)
    return s

df_schemas["schema_gold"] = df_schemas["IMAGE_SCHEMA_ANNOTATION"].apply(normalize_schema_label)
df_eval_fn["schema_gold"] = df_eval_fn["label"].apply(normalize_schema_label)

print("Schèmes du corpus 1 non couverts par SCHEMA_DEFINITIONS :",
      set(df_schemas["schema_gold"].dropna().unique()) - set(SCHEMA_DEFINITIONS))
print("Schèmes du corpus 2 non couverts par SCHEMA_DEFINITIONS :",
      set(df_eval_fn["schema_gold"].dropna().unique()) - set(SCHEMA_DEFINITIONS))


## 6. Construction de l'ensemble de test soumis aux LLMs

Pour maîtriser le coût/temps des appels API, on tire un échantillon stratifié (par schème et par
langue) du corpus 1, et on garde l'intégralité du corpus 2 (98 expressions, léger, avec baseline).
`N_PER_CLASS` est volontairement petit par défaut (mode démo / premiers tests) — augmentez-le pour
l'évaluation finale du mémoire.


In [ ]:
N_PER_CLASS = 3  # nb d'exemples tirés par (schème, langue) dans le corpus 1

def stratified_sample(df, group_cols, n_per_group, seed=RANDOM_SEED):
    parts = [g.sample(min(len(g), n_per_group), random_state=seed) for _, g in df.groupby(group_cols)]
    return pd.concat(parts).reset_index(drop=True)

sample_schemas = stratified_sample(
    df_schemas.dropna(subset=["schema_gold"]),
    group_cols=["schema_gold", "Language"],
    n_per_group=N_PER_CLASS,
)

test_set_corpus1 = sample_schemas[["LinguisticExamples", "schema_gold", "Language"]].rename(
    columns={"LinguisticExamples": "expression"}
)
test_set_corpus1["source_corpus"] = "image_schemas_en_de"

test_set_corpus2 = df_eval_fn[["tweet_text", "schema_gold", "pred"]].rename(
    columns={"tweet_text": "expression", "pred": "baseline_pred"}
)
test_set_corpus2["Language"] = "en"
test_set_corpus2["source_corpus"] = "eval_fnroles"

print(f"Échantillon corpus 1 : {len(test_set_corpus1)} expressions")
print(f"Corpus 2 (intégral)  : {len(test_set_corpus2)} expressions")
test_set_corpus1.head(3)


## 7. Fonctions d'appel aux LLMs

Une fonction générique par fournisseur, plus un routeur `call_llm(model_cfg, prompt)`. Les SDKs
sont importés à l'intérieur des fonctions (pas au niveau du module) pour que le mode démo
fonctionne même sans ces paquets installés. Un cache disque (`results/llm_cache.json`) évite de
repayer les mêmes requêtes si vous relancez le notebook.


In [ ]:
CACHE_PATH = Path("results/llm_cache.json")
CACHE_PATH.parent.mkdir(exist_ok=True)
_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def _cache_key(model_name, strategy, expression):
    return f"{model_name}|||{strategy}|||{expression}"

def _save_cache():
    CACHE_PATH.write_text(json.dumps(_cache, ensure_ascii=False, indent=1))


def mock_llm_call(model_cfg, prompt, expression=None, gold=None):
    """Réponse simulée pour DEMO_MODE : correcte la plupart du temps, avec un peu de bruit
    réaliste (confusions entre schèmes proches), pour pouvoir tester tout le pipeline sans clé API."""
    rng = random.Random(hash((model_cfg["name"], prompt)) & 0xFFFFFFFF)
    confusable = {
        "CONTAINMENT": ["PART_WHOLE", "CENTER_PERIPHERY"],
        "PART_WHOLE": ["CONTAINMENT", "SUBSTANCE"],
        "SOURCE_PATH_GOAL": ["LOCOMOTION", "FORCE"],
        "SUPPORT": ["CONTACT", "FORCE"],
        "BLOCKAGE": ["FORCE", "SUPPORT"],
        "CENTER_PERIPHERY": ["CONTAINMENT", "PART_WHOLE"],
    }
    chosen = gold if (gold and rng.random() < 0.72) else rng.choice(
        confusable.get(gold, list(SCHEMA_DEFINITIONS)) or list(SCHEMA_DEFINITIONS)
    )
    if "SCHEMA:" in prompt and "JUSTIFICATION" in prompt:
        return f"SCHEMA: {chosen}\nJUSTIFICATION: The expression evokes {chosen.lower().replace('_', ' ')} through its spatial/dynamic framing."
    if "Answer with only the schema name" in prompt:
        return chosen
    return (f"This expression seems to rely on the {chosen.replace('_', ' ').lower()} image schema: "
            f"it frames the situation through a spatial or dynamic configuration typical of that schema.")


def call_openai(model, prompt, **kw):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content


def call_anthropic(model, prompt, **kw):
    import anthropic
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    resp = client.messages.create(
        model=model, max_tokens=512, temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text


def call_mistral(model, prompt, **kw):
    from mistralai import Mistral
    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
    resp = client.chat.complete(
        model=model, temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content


def call_openai_compatible(model, prompt, base_url, api_key_env, **kw):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ.get(api_key_env, ""), base_url=base_url)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content


PROVIDER_FUNCS = {
    "openai": call_openai,
    "anthropic": call_anthropic,
    "mistral": call_mistral,
    "openai_compatible": call_openai_compatible,
}


def call_llm(model_cfg, prompt, strategy, expression, gold=None, max_retries=4):
    key = _cache_key(model_cfg["name"], strategy, expression)
    if key in _cache:
        return _cache[key]

    if DEMO_MODE:
        answer = mock_llm_call(model_cfg, prompt, expression=expression, gold=gold)
    else:
        fn = PROVIDER_FUNCS[model_cfg["provider"]]
        kwargs = {k: v for k, v in model_cfg.items() if k not in ("name", "provider", "model")}
        answer = None
        for attempt in range(max_retries):
            try:
                answer = fn(model_cfg["model"], prompt, **kwargs)
                break
            except Exception as e:
                wait = 2 ** attempt
                print(f"[{model_cfg['name']}] erreur ({e}), retry dans {wait}s...")
                time.sleep(wait)
        if answer is None:
            answer = ""

    _cache[key] = answer
    return answer


## 8. Stratégies de prompting

Trois stratégies, conformément aux objectifs du mémoire :

- **A — Explication libre** : aucune liste fournie, le modèle explique librement.
- **B — Choix guidé** : liste fermée de schèmes (section 5), réponse contrainte au nom du schème.
- **C — Choix guidé + justification** : même liste fermée, mais le modèle doit aussi justifier son
  choix — utile pour l'analyse linguistique qualitative (quels éléments de l'expression le modèle
  associe-t-il au schème ?).


In [ ]:
def build_schema_list_text():
    return "\n".join(f"- {name}: {definition}" for name, definition in SCHEMA_DEFINITIONS.items())

SCHEMA_LIST_TEXT = build_schema_list_text()

def prompt_free_explanation(expression):
    return (
        "You are an expert in cognitive linguistics (image schema theory, Lakoff & Johnson 1980).\n"
        f'Consider the following expression: "{expression}"\n\n'
        "Explain, in your own words, which image schema (or schemas) structures the meaning of this "
        "expression. Describe the spatial or dynamic configuration involved."
    )

def prompt_guided_choice(expression):
    return (
        "You are an expert in cognitive linguistics. Below is a closed list of image schemas with "
        f"short definitions:\n\n{SCHEMA_LIST_TEXT}\n\n"
        f'Consider the expression: "{expression}"\n\n'
        "Which single image schema from the list above best structures the meaning of this "
        "expression? Answer with only the schema name (exactly as written in the list), nothing else."
    )

def prompt_guided_choice_justified(expression):
    return (
        "You are an expert in cognitive linguistics. Below is a closed list of image schemas with "
        f"short definitions:\n\n{SCHEMA_LIST_TEXT}\n\n"
        f'Consider the expression: "{expression}"\n\n'
        "1. Which single image schema from the list best structures the meaning of this expression?\n"
        "2. Justify your answer in 1-3 sentences, identifying which elements of the expression evoke "
        "that schema.\n\n"
        "Answer in exactly this format:\n"
        "SCHEMA: <name>\n"
        "JUSTIFICATION: <text>"
    )

PROMPT_STRATEGIES = {
    "free_explanation": prompt_free_explanation,
    "guided_choice": prompt_guided_choice,
    "guided_choice_justified": prompt_guided_choice_justified,
}


## 9. Extraction / normalisation des réponses

- Stratégies B et C : extraction directe du nom du schème (recherche exacte, puis correspondance
  approchée dans la liste fermée).
- Stratégie A (libre) : repérage heuristique par mots-clés. **Cette extraction est approximative
  par construction** — elle sert de première passe automatique ; l'analyse linguistique qualitative
  du mémoire doit relire un échantillon des réponses libres à la main (voir section 12).


In [ ]:
SCHEMA_KEYWORDS = {
    "CONTAINMENT": ["containment", "container", "inside", "outside", "in and out", "bounded region"],
    "SOURCE_PATH_GOAL": ["source-path-goal", "source path goal", "path", "trajectory", "journey", "goal", "destination"],
    "PART_WHOLE": ["part-whole", "part whole", "part of a whole", "piece", "composed of parts"],
    "CENTER_PERIPHERY": ["center-periphery", "center periphery", "core", "periphery", "central"],
    "SUPPORT": ["support", "holding up", "propping", "bearing the weight"],
    "BLOCKAGE": ["blockage", "obstacle", "barrier", "block", "impede"],
    "OBJECT": ["object schema", "treated as an object", "physical object"],
    "VERTICALITY": ["verticality", "up-down", "up and down", "vertical axis"],
    "FORCE": ["force schema", "compulsion", "causal force"],
    "SUBSTANCE": ["substance", "mass noun", "unbounded material"],
    "SCALE": ["scale schema", "graded", "more/less", "quantity scale"],
    "CONTACT": ["contact schema", "touching"],
    "SPLITTING": ["splitting", "dividing into parts"],
    "COVERING": ["covering schema", "covered by"],
    "LINK": ["link schema", "connection between", "bond"],
    "NEAR_FAR": ["near-far", "near far", "proximity", "distance schema"],
    "FRONT_BACK": ["front-back", "front back"],
    "LOCOMOTION": ["locomotion", "self-propelled"],
}


def extract_schema_guided(response_text):
    """Stratégie B : la réponse est censée être (presque) uniquement le nom du schème."""
    if not response_text:
        return None
    candidate = normalize_schema_label(response_text.strip().splitlines()[0])
    if candidate in SCHEMA_DEFINITIONS:
        return candidate
    for name in SCHEMA_DEFINITIONS:
        if name in normalize_schema_label(response_text):
            return name
    return None


def extract_schema_justified(response_text):
    """Stratégie C : extrait la ligne SCHEMA: ... puis la justification."""
    if not response_text:
        return None, None
    m_schema = re.search(r"SCHEMA:\s*([A-Za-z_\- ]+)", response_text)
    m_just = re.search(r"JUSTIFICATION:\s*(.+)", response_text, re.DOTALL)
    schema = normalize_schema_label(m_schema.group(1)) if m_schema else None
    if schema not in SCHEMA_DEFINITIONS:
        schema = extract_schema_guided(response_text)
    justification = m_just.group(1).strip() if m_just else None
    return schema, justification


def extract_schema_free(response_text):
    """Stratégie A : repérage heuristique par mots-clés (première passe automatique)."""
    if not response_text:
        return []
    text_low = response_text.lower()
    hits = [name for name, kws in SCHEMA_KEYWORDS.items() if any(kw in text_low for kw in kws)]
    return hits


## 10. Boucle d'évaluation

Interroge chaque modèle configuré (section 2), avec chacune des trois stratégies, sur l'ensemble
de test (section 6). Les résultats sont mis en cache disque : relancer la cellule ne refait pas les
requêtes déjà obtenues (utile pour reprendre après une coupure, ou éviter de repayer).


In [ ]:
test_set = pd.concat([test_set_corpus1, test_set_corpus2], ignore_index=True)
print(f"Ensemble de test total : {len(test_set)} expressions "
      f"({len(test_set_corpus1)} corpus 1 + {len(test_set_corpus2)} corpus 2)")

records = []
for model_cfg in MODELS_TO_EVALUATE:
    for strategy_name, prompt_fn in PROMPT_STRATEGIES.items():
        for _, row in tqdm(test_set.iterrows(), total=len(test_set),
                            desc=f"{model_cfg['name']} / {strategy_name}"):
            prompt = prompt_fn(row["expression"])
            raw_response = call_llm(model_cfg, prompt, strategy_name, row["expression"], gold=row["schema_gold"])

            if strategy_name == "guided_choice":
                pred_schema, justification = extract_schema_guided(raw_response), None
            elif strategy_name == "guided_choice_justified":
                pred_schema, justification = extract_schema_justified(raw_response)
            else:
                free_hits = extract_schema_free(raw_response)
                pred_schema = free_hits[0] if len(free_hits) == 1 else (free_hits if free_hits else None)
                justification = None

            records.append({
                "model": model_cfg["name"],
                "strategy": strategy_name,
                "source_corpus": row["source_corpus"],
                "expression": row["expression"],
                "language": row["Language"],
                "gold": row["schema_gold"],
                "raw_response": raw_response,
                "pred_schema": pred_schema,
                "justification": justification,
            })

_save_cache()
results_df = pd.DataFrame(records)
results_df.to_csv("results/llm_evaluation_results.csv", index=False)
print(f"{len(results_df)} réponses collectées -> results/llm_evaluation_results.csv")
results_df.head(5)


## 11. Évaluation quantitative

Pour les stratégies à choix fermé (B, C), on calcule précision/rappel/F1 par schème, par modèle,
et une matrice de confusion. La stratégie A (libre) n'est évaluée quantitativement que sur les cas
où l'extraction heuristique renvoie une unique étiquette (les cas ambigus/multiples sont réservés à
l'analyse qualitative, section 12).


In [ ]:
def single_label(pred):
    if isinstance(pred, list):
        return pred[0] if len(pred) == 1 else None
    return pred

scoreable = results_df.copy()
scoreable["pred_single"] = scoreable["pred_schema"].apply(single_label)
scoreable = scoreable.dropna(subset=["gold", "pred_single"])

for (model, strategy), group in scoreable.groupby(["model", "strategy"]):
    acc = (group["gold"] == group["pred_single"]).mean()
    print(f"{model:15s} | {strategy:24s} | n={len(group):4d} | accuracy={acc:.2%}")


In [ ]:
def plot_confusion(model_name, strategy_name):
    subset = scoreable[(scoreable["model"] == model_name) & (scoreable["strategy"] == strategy_name)]
    if subset.empty:
        print("Pas de données pour", model_name, strategy_name)
        return
    labels = sorted(set(subset["gold"]) | set(subset["pred_single"]))
    cm = confusion_matrix(subset["gold"], subset["pred_single"], labels=labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
    plt.xlabel("Prédit"); plt.ylabel("Gold (annoté)")
    plt.title(f"Matrice de confusion — {model_name} / {strategy_name}")
    plt.tight_layout()
    plt.show()
    print(classification_report(subset["gold"], subset["pred_single"], labels=labels, zero_division=0))

# Exemple : à adapter au(x) modèle(s) réellement évalué(s)
if not scoreable.empty:
    plot_confusion(scoreable["model"].iloc[0], "guided_choice")


In [ ]:
# Comparaison des stratégies de prompting, toutes conditions confondues
strategy_accuracy = (
    scoreable.assign(correct=lambda d: d["gold"] == d["pred_single"])
    .groupby(["strategy", "model"])["correct"].mean()
    .unstack()
)
strategy_accuracy.plot(kind="bar", figsize=(9, 5))
plt.ylabel("Accuracy")
plt.title("Accuracy par stratégie de prompting et par modèle")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
strategy_accuracy


In [ ]:
# Comparaison à la baseline FrameNet/WordNet existante (corpus 2 uniquement, colonne 'pred')
baseline = df_eval_fn.copy()
baseline["baseline_schema_set"] = baseline["pred"].fillna("").apply(
    lambda s: {normalize_schema_label(x) for x in s.split(",") if x.strip()}
)
baseline["baseline_correct"] = baseline.apply(
    lambda r: r["schema_gold"] in r["baseline_schema_set"], axis=1
)
print(f"Baseline (système explicable antérieur) — exact/partial match sur le gold : "
      f"{baseline['baseline_correct'].mean():.2%}  (n={len(baseline)})")

llm_on_corpus2 = scoreable[scoreable["source_corpus"] == "eval_fnroles"]
for (model, strategy), group in llm_on_corpus2.groupby(["model", "strategy"]):
    acc = (group["gold"] == group["pred_single"]).mean()
    print(f"LLM {model:15s} | {strategy:24s} | accuracy={acc:.2%}  vs baseline={baseline['baseline_correct'].mean():.2%}")


## 12. Analyse linguistique qualitative

Objectif du mémoire : ne pas se limiter aux scores, mais caractériser *comment* les modèles
raisonnent — expressions bien traitées, erreurs récurrentes, tendances interprétatives.


In [ ]:
# Erreurs les plus fréquentes (paires gold -> prédit), par modèle
errors = scoreable[scoreable["gold"] != scoreable["pred_single"]]
error_pairs = (
    errors.groupby(["model", "gold", "pred_single"])
    .size().reset_index(name="count")
    .sort_values("count", ascending=False)
)
error_pairs.head(15)


In [ ]:
# Table d'inspection : expressions où TOUS les modèles se trompent (erreurs "dures")
pivot_pred = scoreable.pivot_table(index=["expression", "gold"], columns="model",
                                    values="pred_single", aggfunc="first")
hard_cases = pivot_pred[pivot_pred.apply(
    lambda row: all(v != row.name[1] for v in row.dropna()) and row.notna().any(), axis=1
)]
hard_cases


In [ ]:
# Accord inter-modèles : à quelle fréquence les modèles s'accordent-ils entre eux (indépendamment du gold) ?
agreement = (
    scoreable[scoreable["strategy"] == "guided_choice"]
    .pivot_table(index="expression", columns="model", values="pred_single", aggfunc="first")
)
n_models = agreement.shape[1]
full_agreement_rate = (agreement.nunique(axis=1) == 1).mean() if n_models > 1 else float("nan")
print(f"Accord total inter-modèles (choix guidé) : {full_agreement_rate:.2%} des expressions "
      f"({n_models} modèle(s) comparés)")


In [ ]:
# Lecture qualitative des justifications (stratégie C) pour un schème donné
schema_to_inspect = "CONTAINMENT"
sample_justifications = scoreable[
    (scoreable["strategy"] == "guided_choice_justified") &
    (scoreable["gold"] == schema_to_inspect)
][["model", "expression", "pred_single", "justification"]]
sample_justifications.head(10)


## 13. Ambiguïté et préférences schématiques

Deux angles, conformément au sujet :

1. **Cas multi-interprétables** : dans le corpus 2, la colonne `pred` du système FrameNet/WordNet
   antérieur liste parfois plusieurs schèmes candidats pour une même expression — un indice
   d'ambiguïté schématique. On regarde si les LLMs convergent vers le gold, vers une des
   alternatives du système antérieur, ou divergent complètement.
2. **Directionnalité de SOURCE-PATH-GOAL appliqué au temps** : quand une expression temporelle
   mobilise ce schème, la ligne du temps a-t-elle une direction fixe ? Le temps est-il conçu comme
   mobile (*moving time*) ou est-ce l'observateur qui avance sur une ligne du temps fixe
   (*moving ego*) ? Sondage ciblé ci-dessous.


In [ ]:
# 13.1 Cas ambigus du corpus 2 : le système antérieur propose plusieurs schèmes
ambiguous_cases = baseline[baseline["baseline_schema_set"].apply(len) > 1][
    ["tweet_text", "schema_gold", "pred"]
]
print(f"{len(ambiguous_cases)} expressions signalées comme ambiguës par le système antérieur "
      f"(sur {len(baseline)})")
ambiguous_cases.head(10)


In [ ]:
# Pour ces cas ambigus : les LLMs choisissent-ils le gold, une alternative du système antérieur, ou autre chose ?
ambiguous_expr = set(ambiguous_cases["tweet_text"])
llm_on_ambiguous = scoreable[
    (scoreable["strategy"] == "guided_choice") & (scoreable["expression"].isin(ambiguous_expr))
].merge(baseline[["tweet_text", "baseline_schema_set"]], left_on="expression", right_on="tweet_text")

def classify_choice(row):
    if row["pred_single"] == row["gold"]:
        return "gold"
    if row["pred_single"] in row["baseline_schema_set"]:
        return "alternative_baseline"
    return "autre"

if not llm_on_ambiguous.empty:
    llm_on_ambiguous["choice_type"] = llm_on_ambiguous.apply(classify_choice, axis=1)
    print(llm_on_ambiguous.groupby(["model", "choice_type"]).size().unstack(fill_value=0))


In [ ]:
# 13.2 Sondage ciblé : directionnalité de SOURCE-PATH-GOAL appliqué au temps

def prompt_temporal_direction(expression):
    return (
        f'Consider the expression: "{expression}"\n'
        "This expression applies the SOURCE-PATH-GOAL image schema to time.\n\n"
        "a) Does the timeline evoked here have a fixed, absolute direction, or is its direction "
        "relative to a deictic center (the speaker's 'now')?\n"
        "b) Is TIME conceived as moving (towards a stationary observer), or is the OBSERVER/EGO "
        "conceived as moving along a stationary timeline? (Moving Time vs Moving Ego metaphor)\n"
        "c) In this expression, where does the future lie: ahead/in front, behind, up, down, left, "
        "or right?\n\n"
        "Answer in exactly this format:\n"
        "DIRECTION_TYPE: <fixed | relative>\n"
        "METAPHOR: <moving_time | moving_ego | unclear>\n"
        "FUTURE_LOCATION: <ahead | behind | up | down | left | right | other>\n"
        "EXPLANATION: <1-2 sentences>"
    )

# Expressions temporelles candidates : sous-échantillon SOURCE_PATH_GOAL des deux corpus
temporal_candidates = test_set[
    (test_set["schema_gold"] == "SOURCE_PATH_GOAL") &
    (test_set["expression"].str.contains(
        r"\b(?:year|time|deadline|future|past|life|midnight|moment|forward|behind)\b",
        case=False, regex=True))
]
print(f"{len(temporal_candidates)} expressions temporelles candidates pour le sondage de directionnalité")
temporal_candidates[["expression", "source_corpus"]]


In [ ]:
def parse_temporal_response(text):
    def grab(field):
        m = re.search(rf"{field}:\s*(.+)", text or "", re.IGNORECASE)
        return m.group(1).strip().split("\n")[0] if m else None
    return {
        "direction_type": grab("DIRECTION_TYPE"),
        "metaphor": grab("METAPHOR"),
        "future_location": grab("FUTURE_LOCATION"),
        "explanation": grab("EXPLANATION"),
    }

temporal_records = []
for model_cfg in MODELS_TO_EVALUATE:
    for _, row in temporal_candidates.iterrows():
        prompt = prompt_temporal_direction(row["expression"])
        raw = call_llm(model_cfg, prompt, "temporal_direction", row["expression"], gold=row["schema_gold"])
        parsed = parse_temporal_response(raw)
        temporal_records.append({"model": model_cfg["name"], "expression": row["expression"], **parsed})

_save_cache()
temporal_df = pd.DataFrame(temporal_records)
temporal_df.to_csv("results/temporal_direction_results.csv", index=False)
temporal_df


In [ ]:
if not temporal_df.empty:
    print("Répartition METAPHOR (moving time / moving ego) par modèle :")
    print(temporal_df.groupby("model")["metaphor"].value_counts())
    print()
    print("Répartition FUTURE_LOCATION par modèle :")
    print(temporal_df.groupby("model")["future_location"].value_counts())


## 14. Sauvegarde et pistes pour la rédaction du mémoire

- `results/llm_evaluation_results.csv` — toutes les réponses brutes + étiquettes extraites, pour
  les 3 stratégies × modèles configurés. Base pour les tableaux quantitatifs du mémoire.
- `results/temporal_direction_results.csv` — réponses au sondage de directionnalité temporelle,
  pour la discussion sur les préférences schématiques face à l'ambiguïté (objectif 5 du sujet).
- `results/llm_cache.json` — cache brut des appels (permet de reprendre l'évaluation sans repayer).

**Suggestions pour la suite du mémoire :**
1. Passer `DEMO_MODE = False`, renseigner de vraies clés, et augmenter `N_PER_CLASS` pour
   l'évaluation finale (actuellement volontairement petit pour un test rapide/gratuit).
2. Relire manuellement un échantillon des réponses en *explication libre* (stratégie A) — c'est là
   que l'analyse linguistique la plus riche se joue, l'extraction par mots-clés n'étant qu'un tri
   automatique de première passe.
3. Étendre le sondage de directionnalité temporelle (section 13.2) à davantage d'expressions et
   comparer EN vs DE (le corpus 1 est bilingue).
4. Envisager d'ajouter d'autres modèles (open source via Hugging Face/Ollama) en complétant
   `MODELS_TO_EVALUATE` avec `provider: "openai_compatible"` et un `base_url` local.
